<a href="https://colab.research.google.com/github/23f2003543/nlp-lab-programs/blob/main/NLP_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP LAB Programs 1 to 8

## Program 8
Implement the machine translation application of NLP where it needs to train a machine translation model for a language with limited parallel corpora. Investigate and incorporate techniques to improve performance in low-resource scenarios.

In [12]:
import nltk
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Dense, Input
from tensorflow.keras.models import Model

nltk.download('punkt')
def load_parallel_corpos(dataset_name):
  source_text = ["hello world", "how are you", "good morning"]
  target_text = ["hola mundo", "como estas", "buenos dias"]
  return source_text, target_text
def preprocess_sentences(sentences):
  return [nltk.word_tokenize(sent.lower()) for sent in sentences]
def train_translation_model(src, tgt):
  return lambda x:x
source_sentences, target_sentences = load_parallel_corpos("low_resource_dataset")
preprocessed_source = preprocess_sentences(source_sentences)
preprocessed_target = preprocess_sentences(target_sentences)
reverse_model = train_translation_model(preprocessed_source, preprocessed_target)
synthetic_source_sentence = [reverse_model(tgt) for tgt in preprocessed_target]
augmented_source = preprocessed_source + synthetic_source_sentence
augmented_target = preprocessed_target + preprocessed_source
source_tokenizer = Tokenizer()
source_tokenizer.fit_on_texts(augmented_source)
source_vocab_size = len(source_tokenizer.word_index)+1
target_tokenizer = Tokenizer()
target_tokenizer.fit_on_texts(augmented_target)
target_vocab_size = len(target_tokenizer.word_index)+1
source_sequences = source_tokenizer.texts_to_sequences(augmented_source)
target_sequences = target_tokenizer.texts_to_sequences(augmented_target)
max_length = max(max(len(seq) for seq in source_sequences), max(len(seq) for seq in target_sequences))
source_sequences = pad_sequences(source_sequences, maxlen=max_length, padding='post')
target_sequences = pad_sequences(target_sequences, maxlen=max_length, padding='post')
embending_dim = 25
hidden_size = 256
encoder_inputs = Input(shape=(max_length,))
enc_embedding = Embedding(source_vocab_size, embending_dim, mask_zero=True)(encoder_inputs)
encoder_lstm = LSTM(hidden_size, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(enc_embedding)
encoder_states = [state_h, state_c]
decoder_inputs = Input(shape=(max_length,))
dec_embedding = Embedding(target_vocab_size, embending_dim, mask_zero=True)(decoder_inputs)
decoder_lstm = LSTM(hidden_size, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_embedding, initial_state=encoder_states)
decoder_dense = Dense(target_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
epochs = 10
y_train = np.expand_dims(target_sequences, -1)
model.fit([source_sequences, target_sequences], y_train, epochs=epochs)
def translate_sentence(sentence):
  seq = source_tokenizer.texts_to_sequences([nltk.word_tokenize(sentence.lower())])
  seq = pad_sequences(seq, maxlen=max_length, padding='post')
  prediction = model.predict([seq, seq])
  predicted_ids = np.argmax(prediction[0], axis=-1)
  return ' '.join([target_tokenizer.index_word[idx] for idx in predicted_ids if idx > 0])
test_sentence = "hello world"
translation = translate_sentence(test_sentence)
print("Source:", test_sentence)
print("Translation:", translation)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 2.6407
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 2.6358
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 2.6307
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 2.6256
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - loss: 2.6202
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - loss: 2.6146
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 2.6086
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - loss: 2.6021
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - loss: 2.5950
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 2.5872
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 502ms/step
Source: hello world
Translation: hola mundo mundo


## Program 7
 Write a Python program to find synonyms and antonyms of the word "active" using WordNet.

In [14]:
import nltk
from nltk.corpus import wordnet
nltk.download('wordnet')
word = "active"
synonyms=set()
antonyms=set()
for synset in wordnet.synsets(word):
  for lemma in synset.lemmas():
    synonyms.add(lemma.name())
    for antonym in lemma.antonyms():
      antonyms.add(antonym.name())
print("Synonyms of 'active':", synonyms)
print("Antonyms of 'active':", antonyms)


[nltk_data] Downloading package wordnet to /root/nltk_data...


Synonyms of 'active': {'fighting', 'active_agent', 'combat-ready', 'active', 'alive', 'participating', 'active_voice', 'dynamic'}
Antonyms of 'active': {'stative', 'extinct', 'passive', 'quiet', 'inactive', 'dormant', 'passive_voice'}


## Program 6
 Demonstrate the following using appropriate programming tool which illustrates the use
of information retrieval in NLP:
● Study the various Corpus – Brown, Inaugural, Reuters, udhr with various methods like
filelds, raw, words, sents, categories 3
● Create and use your own corpora (plaintext, categorical)
● Study Conditional frequency distributions
● Study of tagged corpora with methods like tagged_sents, tagged_words
● Write a program to find the most frequent noun tags
● Map Words to Properties Using Python Dictionaries
● Study Rule based tagger, Unigram Tagger
Find different words from a given plain text without any space by comparing this text with a
given corpus of words. Also find the score of words.

In [17]:
import nltk
from nltk.corpus import brown, inaugural, reuters, udhr
from nltk.probability import ConditionalFreqDist
from nltk.tag import pos_tag
nltk.download('brown')
nltk.download('inaugural')
nltk.download('reuters')
nltk.download('udhr')
print("Brown Corpus File IDs:", brown.fileids())
print("Inaugural Corpus File IDs:", inaugural.fileids())
print("Reuters Categories:", reuters.categories())
print("UDHR Raw (English) Snippet:", udhr.raw('English-Latin1')[:100])
print("First 20 words of Brown Corpus:", brown.words()[:20])
print("First 5 sentences of Inaugural Corpus:", inaugural.sents()[:5])
corpus_custom = {
    "news": "Today the stock market soared as investors cheered positive earnings.",
    "sports": "The local team won the championship in a thrilling final game."
}
print("Custom Corpus Categories:", corpus_custom.keys())
for category, text in corpus_custom.items():
  print(f"Category: {category}, Sample Text: {text}")
cfd = ConditionalFreqDist()
for category in brown.categories():
  for word in brown.words(categories=category):
    cfd[category][word.lower()] += 1
print("Common words in 'news':", cfd["news"].most_common(5))
print("Common words in 'romance':", cfd["romance"].most_common(5))
news_tagged_sents = brown.tagged_sents(categories="news")
news_tagged_words = brown.tagged_words(categories="news")
print("First 5 tagged sentences in 'news':", news_tagged_sents[:5])
print("First 10 tagged words in 'news':", news_tagged_words[:10])
noun_tag_freq = {}
for word, tag in news_tagged_words:
  if tag.startswith("NN"):
    noun_tag_freq[tag] = noun_tag_freq.get(tag, 0) + 1
most_frequent_noun_tag = max(noun_tag_freq, key=noun_tag_freq.get)
print("Most frequent noun tag:", most_frequent_noun_tag, "with count:", noun_tag_freq[most_frequent_noun_tag])
sample_text = "This is a sample text with sample words and sample analysis"
words_list = sample_text.split()
word_properties = {}
for word in words_list:
  word_lower = word.lower()
  if word_lower not in word_properties:
    word_properties[word_lower] = {"length": len(word_lower), "frequency": 1}
  else:
    word_properties[word_lower]["frequency"] += 1
print("Word Properties:", word_properties)
def segment_text(text, valid_words):
  if not text:
    return [[]]
  segmentation_results = []
  for i in range(1, len(text) + 1):
    prefix = text[:i]
    if prefix in valid_words:
      suffix_results = segment_text(text[i:], valid_words)
      for segmentation in suffix_results:
        segmentation_results.append([prefix] + segmentation)
  return segmentation_results
valid_words = ["hello", "this", "is", "a", "test", "his", "s", "isatest"]
input_text = "hellothisisatest"
segmentations = segment_text(input_text, valid_words)
print("Possible segmentations for", input_text)
for segmentation in segmentations:
  score = 1 / len(segmentation)
  print(segmentation, "Score:", score)

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package inaugural to /root/nltk_data...
[nltk_data]   Package inaugural is already up-to-date!
[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package udhr to /root/nltk_data...
[nltk_data]   Package udhr is already up-to-date!


Brown Corpus File IDs: ['ca01', 'ca02', 'ca03', 'ca04', 'ca05', 'ca06', 'ca07', 'ca08', 'ca09', 'ca10', 'ca11', 'ca12', 'ca13', 'ca14', 'ca15', 'ca16', 'ca17', 'ca18', 'ca19', 'ca20', 'ca21', 'ca22', 'ca23', 'ca24', 'ca25', 'ca26', 'ca27', 'ca28', 'ca29', 'ca30', 'ca31', 'ca32', 'ca33', 'ca34', 'ca35', 'ca36', 'ca37', 'ca38', 'ca39', 'ca40', 'ca41', 'ca42', 'ca43', 'ca44', 'cb01', 'cb02', 'cb03', 'cb04', 'cb05', 'cb06', 'cb07', 'cb08', 'cb09', 'cb10', 'cb11', 'cb12', 'cb13', 'cb14', 'cb15', 'cb16', 'cb17', 'cb18', 'cb19', 'cb20', 'cb21', 'cb22', 'cb23', 'cb24', 'cb25', 'cb26', 'cb27', 'cc01', 'cc02', 'cc03', 'cc04', 'cc05', 'cc06', 'cc07', 'cc08', 'cc09', 'cc10', 'cc11', 'cc12', 'cc13', 'cc14', 'cc15', 'cc16', 'cc17', 'cd01', 'cd02', 'cd03', 'cd04', 'cd05', 'cd06', 'cd07', 'cd08', 'cd09', 'cd10', 'cd11', 'cd12', 'cd13', 'cd14', 'cd15', 'cd16', 'cd17', 'ce01', 'ce02', 'ce03', 'ce04', 'ce05', 'ce06', 'ce07', 'ce08', 'ce09', 'ce10', 'ce11', 'ce12', 'ce13', 'ce14', 'ce15', 'ce16', 'ce17', 

## Program 5
Given the following short movie reviews, each labeled with a genre, either comedy or action:
● fun, couple, love, love comedy ● fast, furious, shoot action ● couple, fly, fast, fun, fun comedy
● furious, shoot, shoot, fun action ● fly, fast, shoot, love action and A new document D: fast,
couple, shoot, fly Compute the most likely class for D. Assume a Naive Bayes classifier and use
add-1 smoothing for the likelihoods.

In [19]:
from collections import Counter
reviews = [
    ("fun, couple, love, love", "comedy"),
    ("fast, furious, shoot", "action"),
    ("couple, fly, fast, fun, fun", "comedy"),
    ("furious, shoot, shoot, fun", "action"),
    ("fly, fast, shoot, love", "action")
]
new_document = "fast, couple, shoot, fly"
def preprocess_text(text):
  return text.lower().split(", ")
class_word_counts = {"comedy": Counter(), "action": Counter()}
class_doc_counts = {"comedy": 0, "action": 0}
for review, genre in reviews:
  class_doc_counts[genre] += 1
  words = preprocess_text(review)
  class_word_counts[genre].update(words)
total_docs = len(reviews)
prior_comedy = class_doc_counts["comedy"] / total_docs
prior_action = class_doc_counts["action"] / total_docs
all_words = set()
for review, genre in reviews:
  words = preprocess_text(review)
  all_words.update(words)
vocab_size = len(all_words)
def calculate_likelihood(word, genre):
  word_count = class_word_counts[genre][word]
  total_words_in_class = sum(class_word_counts[genre].values())
  return (word_count + 1) / (total_words_in_class + vocab_size)
def classify_document(new_document):
  words_in_document = preprocess_text(new_document)
  log_prob_comedy = 0
  log_prob_action = 0
  log_prob_comedy += prior_comedy
  log_prob_action += prior_action
  for word in words_in_document:
    likelihood_comedy = calculate_likelihood(word, "comedy")
    likelihood_action = calculate_likelihood(word, "action")
    log_prob_comedy += likelihood_comedy
    log_prob_action += likelihood_action
  if log_prob_comedy > log_prob_action:
    return "comedy"
  else:
    return "action"
result = classify_document(new_document)
print(f"The predicted class for the new document is: {result}")

The predicted class for the new document is: action


## Program 4
 Write a program to implement top-down and bottom-up parser using appropriate context free grammar.


In [21]:
class TopDownParser:
  def __init__(self, grammar):
    self.grammar = grammar
    self.input_string = ""
    self.pos = 0
  def parse(self, input_string):
    self.input_string = input_string
    self.pos = 0
    return self.E()
  def E(self):
    result = self.T()
    if result is None:
      return None
    while self.pos < len(self.input_string) and self.input_string[self.pos]=='+':
      self.pos += 1
      result2 = self.T()
      if result2 is None:
        return None
    return result
  def T(self):
    result = self.F()
    if result is None:
      return None
    while self.pos < len(self.input_string) and self.input_string[self.pos] == '*':
      self.pos += 1
      result2 = self.F()
      if result2 is None:
        return None
    return result
  def F(self):
    if self.pos < len(self.input_string) and self.input_string[self.pos] == '(':
      self.pos += 1
      result = self.E()
      if result is None or self.pos >= len(self.input_string) or self.input_string[self.pos]!=')':
        return None
      self.pos +=1
      return result
    elif self.pos < len(self.input_string) and self.input_string[self.pos].isalpha():
      self.pos += 1
      return True
    return None
topdown_parser = TopDownParser(grammar=None)
input_expr = "id+id*id"
print("Top-Down Parser Result:", topdown_parser.parse(input_expr))


Top-Down Parser Result: True


In [22]:
class BottomUpParser:
  def __init__(self, grammar):
    self.grammar = grammar
    self.stack = []
    self.input_string = ""
    self.pos = 0
  def parse(self, input_string):
    self.input_string = input_string
    self.pos = 0
    while self.pos < len(self.input_string):
      self.shift()
      while self.reduce():
        pass
    return self.stack == ['S']
  def shift(self):
    if self.pos < len(self.input_string):
      self.stack.append(self.input_string[self.pos])
      self.pos += 1
  def reduce(self):
    if len(self.stack) >= 3 and self.stack[-3:] == ['id','+','id']:
      self.stack = self.stack[:-3]
      self.stack.append('E')
      return True
    elif len(self.stack) >= 3 and self.stack[-3:] == ['id','*','id']:
      self.stack = self.stack[-3:]
      self.stack.append('T')
      return True
    elif len(self.stack) >= 1 and self.stack[-1]=='id':
      self.stack = self.stack[:-1]
      self.stack.append('F')
      return True
    return False
bottomup_parser = BottomUpParser(grammar=None)
input_expr = "id+id*id"
print("Bottom-Up Parser Result:", bottomup_parser.parse(input_expr))

Bottom-Up Parser Result: False


## Program 3
Investigate the Minimum Edit Distance (MED) algorithm and its application in string
comparison and the goal is to understand how the algorithm efficiently computes the minimum
number of edit operations required to transform one string into another. ● Test the algorithm on
strings with different type of variations (e.g., typos, substitutions, insertions, deletions) ●
Evaluate its adaptability to different types of input variations

In [23]:
def min_edit_distance(X,Y):
  m,n = len(X), len(Y)
  D = [[0]*(n+1) for _ in range(m+1)]
  for i in range(m+1):
    D[i][0] = i
  for j in range(n+1):
    D[0][j] = j
  for i in range(1, m+1):
    for j in range(1, n+1):
      if X[i-1] == Y[j-1]:
        D[i][j] = D[i-1][j-1]
      else:
        D[i][j] = min(D[i][j-1], D[i-1][j], D[i-1][j-1])+1
  return D[m][n]
print(min_edit_distance("kitten", "sitten")) # Output: 1 (substitution)
print(min_edit_distance("kitten", "kittens")) # Output: 1 (insertion)
print(min_edit_distance("kitten", "kiten")) # Output: 1 (deletion)
print(min_edit_distance("sitting", "kitten")) # Output: 3 (substitution + substitution + deletion)
print(min_edit_distance("kitten", "kitten")) # Output: 0 (no changes)
print(min_edit_distance("abc", "xyz")) #subsitution 3

1
1
1
3
0
3


## Program 2
Demonstrate the N-gram modeling to analyze and establish the probability distribution across
sentences and explore the utilization of unigrams, bigrams, and trigrams in diverse English
sentences to illustrate the impact of varying n-gram orders on the calculated probabilities.

In [27]:
from collections import Counter
import random
sentence = [
    "I love ice cream",
    "I love chocolate",
    "Ice cream is delicious"
]
words = []
for sentence in sentence:
  words.extend(sentence.lower().split())
total_words = len(words)
unigrams = Counter(words)
def calculate_unigram_probabilities(unigrams, total_words):
  unigram_probabilities = {word:count/total_words for word, count in unigrams.items()}
  return unigram_probabilities
bigrams = [(words[i], words[i+1]) for i in range(len(words)-1)]
bigram_counts = Counter(bigrams)
def calculate_bigram_probabilities(bigrams, bigram_count, unigrams):
  bigram_probabilities = {}
  for (word1, word2), count in bigram_counts.items():
    bigram_probabilities[(word1, word2)] = count/unigrams[word1]
  return bigram_probabilities
trigrams = [(words[i], words[i+1], words[i+2]) for i in range(len(words)-2)]
trigram_counts = Counter(trigrams)
def calculate_trigram_probabilities(trigrams, trigram_counts, bigrams):
  trigram_probabilities = {}
  for (word1, word2, word3), count in trigram_counts.items():
    bigram_count = bigrams[(word1, word2)]
    trigram_probabilities[(word1, word2, word3)] = count/bigram_count
  return trigram_probabilities
unigram_probabilities = calculate_unigram_probabilities(unigrams, total_words)
bigram_probabilities = calculate_bigram_probabilities(bigrams, bigram_counts, unigrams)
trigram_probabilities = calculate_trigram_probabilities(trigrams, trigram_counts, bigram_counts)
print("Unigram Probabilities:")
for word, prob in unigram_probabilities.items():
  print(f"P({word}) = {prob:.4f}")

print("\nBigram Probabilities:")
for (word1, word2), prob in bigram_probabilities.items():
  print(f"P({word2} | {word1}) = {prob:.4f}")

print("\nTrigram Probabilities:")
for (word1, word2, word3), prob in trigram_probabilities.items():
  print(f"P({word3} | {word1}, {word2}) = {prob:.4f}")

Unigram Probabilities:
P(i) = 0.1818
P(love) = 0.1818
P(ice) = 0.1818
P(cream) = 0.1818
P(chocolate) = 0.0909
P(is) = 0.0909
P(delicious) = 0.0909

Bigram Probabilities:
P(love | i) = 1.0000
P(ice | love) = 0.5000
P(cream | ice) = 1.0000
P(i | cream) = 0.5000
P(chocolate | love) = 0.5000
P(ice | chocolate) = 1.0000
P(is | cream) = 0.5000
P(delicious | is) = 1.0000

Trigram Probabilities:
P(ice | i, love) = 0.5000
P(cream | love, ice) = 1.0000
P(i | ice, cream) = 0.5000
P(love | cream, i) = 1.0000
P(chocolate | i, love) = 0.5000
P(ice | love, chocolate) = 1.0000
P(cream | chocolate, ice) = 1.0000
P(is | ice, cream) = 0.5000
P(delicious | cream, is) = 1.0000


## Program 1
Write a Python program for the following preprocessing of text in NLP:
● Tokenization
● Filtration
● Script Validation
● Stop Word Removal
● Stemming

In [26]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('punkt')
nltk.download('stopwords')
def preprocess_text(text):
  tokens = nltk.word_tokenize(text)
  print("Tokens:", tokens)
  filtered_tokens = [word for word in tokens if re.match(r'^[a-zA-Z]+$', word)]
  print("Filtered Tokens:", filtered_tokens)
  english_tokens = [word for word in filtered_tokens if word.isalpha()]
  print("English Tokens:", english_tokens)
  stop_words = set(stopwords.words('english'))
  cleaned_tokens = [word for word in english_tokens if word.lower() not in stop_words]
  print("Tokens after Stop Word Removal:", cleaned_tokens)
  stemmer = PorterStemmer()
  stemmed_tokens = [stemmer.stem(word) for word in cleaned_tokens]
  print("Stemmed Tokens:", stemmed_tokens)
  return stemmed_tokens
text = "The quick brown fox jumps over the lazy dog! 123"
preprocessed_text = preprocess_text(text)
print("Final Preprocessed Text:", preprocessed_text)

Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '!', '123']
Filtered Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
English Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
Tokens after Stop Word Removal: ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog']
Stemmed Tokens: ['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']
Final Preprocessed Text: ['quick', 'brown', 'fox', 'jump', 'lazi', 'dog']


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
